In [ ]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/iec/MinhHieu/rPPG"


## Inlined source

The cells below contain the model / loss source that was previously imported from the `neural_methods/` (and `evaluation/`) packages. They are inlined here so the notebook is self-contained.


In [ ]:
# === inlined from neural_methods/model/PhysFormer.py ===
"""This file is a combination of Physformer.py and transformer_layer.py
   in the official PhysFormer implementation here:
   https://github.com/ZitongYu/PhysFormer

   model.py - Model and module class for ViT.
   They are built to mirror those in the official Jax implementation.
"""

import numpy as np
from typing import Optional
import torch
from torch import nn
from torch import Tensor 
from torch.nn import functional as F
import math

def as_tuple(x):
    return x if isinstance(x, tuple) else (x, x)

'''
Temporal Center-difference based Convolutional layer (3D version)
theta: control the percentage of original convolution and centeral-difference convolution
'''
class CDC_T(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,
                 padding=1, dilation=1, groups=1, bias=False, theta=0.6):

        super(CDC_T, self).__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding,
                              dilation=dilation, groups=groups, bias=bias)
        self.theta = theta

    def forward(self, x):
        out_normal = self.conv(x)

        if math.fabs(self.theta - 0.0) < 1e-8:
            return out_normal
        else:
            [C_out, C_in, t, kernel_size, kernel_size] = self.conv.weight.shape

            # only CD works on temporal kernel size>1
            if self.conv.weight.shape[2] > 1:
                kernel_diff = self.conv.weight[:, :, 0, :, :].sum(2).sum(2) + self.conv.weight[:, :, 2, :, :].sum(
                    2).sum(2)
                kernel_diff = kernel_diff[:, :, None, None, None]
                out_diff = F.conv3d(input=x, weight=kernel_diff, bias=self.conv.bias, stride=self.conv.stride,
                                    padding=0, dilation=self.conv.dilation, groups=self.conv.groups)
                return out_normal - self.theta * out_diff

            else:
                return out_normal


def split_last(x, shape):
    "split the last dimension to given shape"
    shape = list(shape)
    assert shape.count(-1) <= 1
    if -1 in shape:
        shape[shape.index(-1)] = int(x.size(-1) / -np.prod(shape))
    return x.view(*x.size()[:-1], *shape)


def merge_last(x, n_dims):
    "merge the last n_dims to a dimension"
    s = x.size()
    assert n_dims > 1 and n_dims < len(s)
    return x.view(*s[:-n_dims], -1)

class MultiHeadedSelfAttention_TDC_gra_sharp(nn.Module):
    """Multi-Headed Dot Product Attention with depth-wise Conv3d"""
    def __init__(self, dim, num_heads, dropout, theta):
        super().__init__()
        
        self.proj_q = nn.Sequential(
            CDC_T(dim, dim, 3, stride=1, padding=1, groups=1, bias=False, theta=theta),  
            nn.BatchNorm3d(dim),
        )
        self.proj_k = nn.Sequential(
            CDC_T(dim, dim, 3, stride=1, padding=1, groups=1, bias=False, theta=theta),  
            nn.BatchNorm3d(dim),
        )
        self.proj_v = nn.Sequential(
            nn.Conv3d(dim, dim, 1, stride=1, padding=0, groups=1, bias=False),
        )
        
        self.drop = nn.Dropout(dropout)
        self.n_heads = num_heads
        self.scores = None # for visualization

    def forward(self, x, gra_sharp):    # [B, 4*4*40, 128]
        """
        x, q(query), k(key), v(value) : (B(batch_size), S(seq_len), D(dim))
        mask : (B(batch_size) x S(seq_len))
        * split D(dim) into (H(n_heads), W(width of head)) ; D = H * W
        """
        # (B, S, D) -proj-> (B, S, D) -split-> (B, S, H, W) -trans-> (B, H, S, W)
        
        [B, P, C]=x.shape
        x = x.transpose(1, 2).view(B, C, P//16, 4, 4)      # [B, dim, 40, 4, 4]
        q, k, v = self.proj_q(x), self.proj_k(x), self.proj_v(x)
        q = q.flatten(2).transpose(1, 2)  # [B, 4*4*40, dim]
        k = k.flatten(2).transpose(1, 2)  # [B, 4*4*40, dim]
        v = v.flatten(2).transpose(1, 2)  # [B, 4*4*40, dim]
        
        q, k, v = (split_last(x, (self.n_heads, -1)).transpose(1, 2) for x in [q, k, v])
        # (B, H, S, W) @ (B, H, W, S) -> (B, H, S, S) -softmax-> (B, H, S, S)
        scores = q @ k.transpose(-2, -1) / gra_sharp

        scores = self.drop(F.softmax(scores, dim=-1))
        # (B, H, S, S) @ (B, H, S, W) -> (B, H, S, W) -trans-> (B, S, H, W)
        h = (scores @ v).transpose(1, 2).contiguous()
        # -merge-> (B, S, D)
        h = merge_last(h, 2)
        self.scores = scores
        return h, scores




class PositionWiseFeedForward_ST(nn.Module):
    """FeedForward Neural Networks for each position"""
    def __init__(self, dim, ff_dim):
        super().__init__()
        
        self.fc1 = nn.Sequential(
            nn.Conv3d(dim, ff_dim, 1, stride=1, padding=0, bias=False),  
            nn.BatchNorm3d(ff_dim),
            nn.ELU(),
        )
        
        self.STConv = nn.Sequential(
            nn.Conv3d(ff_dim, ff_dim, 3, stride=1, padding=1, groups=ff_dim, bias=False),  
            nn.BatchNorm3d(ff_dim),
            nn.ELU(),
        )
        
        self.fc2 = nn.Sequential(
            nn.Conv3d(ff_dim, dim, 1, stride=1, padding=0, bias=False),  
            nn.BatchNorm3d(dim),
        )

    def forward(self, x):    # [B, 4*4*40, 128]
        [B, P, C]=x.shape
        x = x.transpose(1, 2).view(B, C, P//16, 4, 4)      # [B, dim, 40, 4, 4]
        x = self.fc1(x)		              # x [B, ff_dim, 40, 4, 4]
        x = self.STConv(x)		          # x [B, ff_dim, 40, 4, 4]
        x = self.fc2(x)		              # x [B, dim, 40, 4, 4]
        x = x.flatten(2).transpose(1, 2)  # [B, 4*4*40, dim]
        
        return x

class Block_ST_TDC_gra_sharp(nn.Module):
    """Transformer Block"""
    def __init__(self, dim, num_heads, ff_dim, dropout, theta):
        super().__init__()
        self.attn = MultiHeadedSelfAttention_TDC_gra_sharp(dim, num_heads, dropout, theta)
        self.proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.pwff = PositionWiseFeedForward_ST(dim, ff_dim)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, gra_sharp):
        Atten, Score = self.attn(self.norm1(x), gra_sharp)
        h = self.drop(self.proj(Atten))
        x = x + h
        h = self.drop(self.pwff(self.norm2(x)))
        x = x + h
        return x, Score

class Transformer_ST_TDC_gra_sharp(nn.Module):
    """Transformer with Self-Attentive Blocks"""
    def __init__(self, num_layers, dim, num_heads, ff_dim, dropout, theta):
        super().__init__()
        self.blocks = nn.ModuleList([
            Block_ST_TDC_gra_sharp(dim, num_heads, ff_dim, dropout, theta) for _ in range(num_layers)])

    def forward(self, x, gra_sharp):
        for block in self.blocks:
            x, Score = block(x, gra_sharp)
        return x, Score

# stem_3DCNN + ST-ViT with local Depthwise Spatio-Temporal MLP
class ViT_ST_ST_Compact3_TDC_gra_sharp(nn.Module):

    def __init__(
        self, 
        name: Optional[str] = None, 
        pretrained: bool = False, 
        patches: int = 16,
        dim: int = 768,
        ff_dim: int = 3072,
        num_heads: int = 12,
        num_layers: int = 12,
        attention_dropout_rate: float = 0.0,
        dropout_rate: float = 0.2,
        representation_size: Optional[int] = None,
        load_repr_layer: bool = False,
        classifier: str = 'token',
        #positional_embedding: str = '1d',
        in_channels: int = 3, 
        frame: int = 160,
        theta: float = 0.2,
        image_size: Optional[int] = None,
    ):
        super().__init__()

        
        self.image_size = image_size  
        self.frame = frame  
        self.dim = dim              

        # Image and patch sizes
        t, h, w = as_tuple(image_size)  # tube sizes
        ft, fh, fw = as_tuple(patches)  # patch sizes, ft = 4 ==> 160/4=40
        gt, gh, gw = t//ft, h // fh, w // fw  # number of patches
        seq_len = gh * gw * gt

        # Patch embedding    [4x16x16]conv
        self.patch_embedding = nn.Conv3d(dim, dim, kernel_size=(ft, fh, fw), stride=(ft, fh, fw))
        
        # Transformer
        self.transformer1 = Transformer_ST_TDC_gra_sharp(num_layers=num_layers//3, dim=dim, num_heads=num_heads, 
                                       ff_dim=ff_dim, dropout=dropout_rate, theta=theta)
        # Transformer
        self.transformer2 = Transformer_ST_TDC_gra_sharp(num_layers=num_layers//3, dim=dim, num_heads=num_heads, 
                                       ff_dim=ff_dim, dropout=dropout_rate, theta=theta)
        # Transformer
        self.transformer3 = Transformer_ST_TDC_gra_sharp(num_layers=num_layers//3, dim=dim, num_heads=num_heads, 
                                       ff_dim=ff_dim, dropout=dropout_rate, theta=theta)
        
        
        
        self.Stem0 = nn.Sequential(
            nn.Conv3d(3, dim//4, [1, 5, 5], stride=1, padding=[0,2,2]),
            nn.BatchNorm3d(dim//4),
            nn.ReLU(inplace=True),
            nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2)),
        )
        
        self.Stem1 = nn.Sequential(
            nn.Conv3d(dim//4, dim//2, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(dim//2),
            nn.ReLU(inplace=True),
            nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2)),
        )
        self.Stem2 = nn.Sequential(
            nn.Conv3d(dim//2, dim, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(dim),
            nn.ReLU(inplace=True),
            nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2)),
        )
           
        self.upsample = nn.Sequential(
            nn.Upsample(scale_factor=(2,1,1)),
            nn.Conv3d(dim, dim, [3, 1, 1], stride=1, padding=(1,0,0)),   
            nn.BatchNorm3d(dim),
            nn.ELU(),
        )
        self.upsample2 = nn.Sequential(
            nn.Upsample(scale_factor=(2,1,1)),
            nn.Conv3d(dim, dim//2, [3, 1, 1], stride=1, padding=(1,0,0)),   
            nn.BatchNorm3d(dim//2),
            nn.ELU(),
        )
 
        self.ConvBlockLast = nn.Conv1d(dim//2, 1, 1,stride=1, padding=0)
        
        
        # Initialize weights
        self.init_weights()
        
    @torch.no_grad()
    def init_weights(self):
        def _init(m):
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)  # _trunc_normal(m.weight, std=0.02)  # from .initialization import _trunc_normal
                if hasattr(m, 'bias') and m.bias is not None:
                    nn.init.normal_(m.bias, std=1e-6)  # nn.init.constant(m.bias, 0)
        self.apply(_init)


    def forward(self, x, gra_sharp):

        # b is batch number, c channels, t frame, fh frame height, and fw frame width
        b, c, t, fh, fw = x.shape
        
        x = self.Stem0(x)
        x = self.Stem1(x)
        x = self.Stem2(x)  # [B, 64, 160, 64, 64]
        
        x = self.patch_embedding(x)  # [B, 64, 40, 4, 4]
        x = x.flatten(2).transpose(1, 2)  # [B, 40*4*4, 64]
        
        
        Trans_features, Score1 =  self.transformer1(x, gra_sharp)  # [B, 4*4*40, 64]
        Trans_features2, Score2 =  self.transformer2(Trans_features, gra_sharp)  # [B, 4*4*40, 64]
        Trans_features3, Score3 =  self.transformer3(Trans_features2, gra_sharp)  # [B, 4*4*40, 64]
        
        # upsampling heads
        #features_last = Trans_features3.transpose(1, 2).view(b, self.dim, 40, 4, 4) # [B, 64, 40, 4, 4]
        features_last = Trans_features3.transpose(1, 2).view(b, self.dim, t//4, 4, 4) # [B, 64, 40, 4, 4]
        
        features_last = self.upsample(features_last)		    # x [B, 64, 7*7, 80]
        features_last = self.upsample2(features_last)		    # x [B, 32, 7*7, 160]
        
        features_last = torch.mean(features_last,3)     # x [B, 32, 160, 4]  
        features_last = torch.mean(features_last,3)     # x [B, 32, 160]    
        rPPG = self.ConvBlockLast(features_last)    # x [B, 1, 160]
        
        rPPG = rPPG.squeeze(1)
        
        return rPPG, Score1, Score2, Score3


In [ ]:
# ----- paths -----RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Normal")PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Normal/groupD")OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/Headmotion/groupD")# ----- video / signal params -----VIDEO_FPS   = 30       # camera frame ratePPG_FS      = 60       # PPG sensor sampling rate (Hz)# ----- PhysFormer preprocessing params -----CHUNK_LENGTH = 160     # frames per clipIMG_H, IMG_W = 128, 128  # resolutionLABEL_TYPE   = "DiffNormalized"  # cumsum in post-processingDATA_FORMAT  = "NCDHW"# ----- device -----DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"print("Device:", DEVICE)os.makedirs(PREPROCESSED_PATH, exist_ok=True)os.makedirs(OUTPUT_DIR, exist_ok=True)print("PREPROCESSED_PATH:", PREPROCESSED_PATH)print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# Models list -- uncomment to run additional pretrained models
MODELS = [
    ("PURE_PhysFormer",              "PhysFormer", "final_model_release/PURE_PhysFormer_DiffNormalized.pth"),
    ("SCAMPS_PhysFormer",          "PhysFormer", "final_model_release/SCAMPS_PhysFormer_DiffNormalized.pth"),
    ("UBFC-rPPG_PhysFormer",       "PhysFormer", "final_model_release/UBFC-rPPG_PhysFormer_DiffNormalized.pth"),
]

# Default to first model
MODEL_NAME, MODEL_TYPE, MODEL_REL_PATH = MODELS[0]
MODEL_PATH = os.path.join(REPO_ROOT, MODEL_REL_PATH)
print(f"Selected model: {MODEL_NAME}")
print(f"Model path: {MODEL_PATH}")

In [ ]:
# Read video framesdef read_video_frames(video_path):    """Read all frames from a video file (MP4, MKV, AVI, etc.).    Returns:        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.    """    cap = cv2.VideoCapture(video_path)    if not cap.isOpened():        raise IOError(f"Cannot open video: {video_path}")    frames = []    while True:        ret, frame = cap.read()        if not ret:            break        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))    cap.release()    if not frames:        raise ValueError(f"Empty video: {video_path}")    return np.stack(frames, axis=0)

In [ ]:
def read_ppg_synced(session_path, num_frames):
    """
    Reads the PPG signal from 'ppg.csv' and resamples it to match the exact 
    timestamps of the video frames from 'frame_timestamps.csv'.
    """
    import pandas as pd
    import numpy as np
    import os
    
    # 1. Read the video frame timestamps
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    # Validation
    if len(frame_df) != num_frames:
        print(f"Warning: Video has {num_frames} frames, but frame_timestamps.csv has {len(frame_df)} rows. Using min count.")
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    # 2. Read the raw PPG data
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    # Clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    
    # 3. Resample (Interpolate)
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)

In [ ]:
# Normalization functions

def diff_normalize_data(data):
    """DiffNormalized: (frame[t+1]-frame[t]) / (frame[t+1]+frame[t]+1e-7), then / std."""
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [ ]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
# Discover subjects and read ground truth heart rate

all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_path = subj_dir

    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
    })
    print(f"  {subj_id}  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

In [ ]:
# Data preprocessing

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    ppg_signal = read_ppg_synced(session_path, T)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Face crop and resize
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)

    # DiffNormalized data: 3 channels
    diff_data = diff_normalize_data(frames_cropped)

    # DiffNormalized label
    label = diff_normalize_label(ppg_signal)

    # Chunk into clips of CHUNK_LENGTH frames
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([diff_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])

    # Save per subject
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, data_clips[chunk_idx])   # (CHUNK_LENGTH, H, W, 3)
        np.save(label_path, label_clips[chunk_idx])  # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

In [ ]:
# PyTorch Dataset + DataLoader

class PhysFormerDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (T,)

        # NDHWC -> NCDHW: transpose (3, 0, 1, 2)
        data = np.transpose(data, (3, 0, 1, 2))  # (3, T, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = PhysFormerDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)
print(f"DataLoader ready: {len(loader)} batches")

In [ ]:
# Post-processing helpers

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp(pred_chunks, label_chunks, fs=30, diff_flag=True):
    """Per-subject BVP post-processing with optional cumsum for DiffNormalized."""
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    if diff_flag:
        pred  = detrend(np.cumsum(pred),  100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred  = detrend(pred,  100)
        label = detrend(label, 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

In [ ]:
# Inference loop -> per-subject results -> aggregate metrics -> export
# Results saved to results_groupD/{model_name}/

# Iterate over all models in the MODELS list
for model_name, model_type, model_rel_path in MODELS:
    model_path = os.path.join(REPO_ROOT, model_rel_path)
    print(f"\n{'='*70}")
    print(f"Model: {model_name}")
    print(f"Path:  {model_path}")
    print(f"{'='*70}")

    # Load PhysFormer model
    model = ViT_ST_ST_Compact3_TDC_gra_sharp(
        image_size=(CHUNK_LENGTH, IMG_H, IMG_W),
        patches=(4, 4, 4),
        dim=96,
        ff_dim=144,
        num_heads=4,
        num_layers=12,
        dropout_rate=0.2,
        theta=0.7,
    )
    
    # Đã thêm weights_only=True để ẩn cảnh báo bảo mật
    state_dict = torch.load(model_path, map_location=DEVICE, weights_only=True)
    
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}
        
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()
    
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model loaded. Total parameters: {num_params:,}")

    # Run inference
    bvp_preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray}
    bvp_labels_dict = {}

    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            data, labels_batch, batch_subjects, batch_chunk_ids = batch

            # data shape: (N, 3, T, H, W) -- already NCDHW from dataset
            data = data.to(DEVICE)

            # CRITICAL: PhysFormer forward requires gra_sharp parameter
            rPPG, _, _, _ = model(data, gra_sharp=2.0)

            # rPPG shape: (N, T)
            rPPG_np   = rPPG.cpu().numpy()
            labels_np = labels_batch.numpy()

            N = data.shape[0]
            for i in range(N):
                subj = batch_subjects[i]
                cid  = int(batch_chunk_ids[i])

                if subj not in bvp_preds_dict:
                    bvp_preds_dict[subj]  = {}
                    bvp_labels_dict[subj] = {}

                bvp_preds_dict[subj][cid]  = rPPG_np[i]
                bvp_labels_dict[subj][cid] = labels_np[i]

    print(f"\nInference complete. Subjects: {sorted(bvp_preds_dict.keys())}")

    # Per-subject results
    per_subject_results = []
    hr_preds_all  = []
    hr_labels_all = []
    snr_all       = []

    FS = VIDEO_FPS

    print(f"\n{'Subject':<10} {'HR_pred':>10} {'HR_label':>10} {'HR_err':>8} {'SNR':>7}")
    print("-" * 55)

    for subj_key in sorted(bvp_preds_dict.keys()):
        hr_pred, hr_label, _, pred_processed = process_bvp(
            bvp_preds_dict[subj_key], bvp_labels_dict[subj_key], fs=FS, diff_flag=True
        )

        snr_db = calculate_snr(pred_processed, hr_label, FS)
        hr_err = hr_pred - hr_label

        # map subj_key ("S000") back to original ID ("S_000")
        subj_id = subj_key[0] + "_" + subj_key[1:]

        per_subject_results.append({
            "name":                subj_id,
            "predicted_heartrate": hr_pred,
            "label_heartrate":     hr_label,
            "heartrate_error":     hr_err,
            "snr_db":              snr_db,
        })

        hr_preds_all.append(hr_pred)
        hr_labels_all.append(hr_label)
        snr_all.append(snr_db)

        print(f"{subj_id:<10} {hr_pred:>10.3f} {hr_label:>10.3f} {hr_err:>8.3f} {snr_db:>7.2f}")

    hr_preds_all  = np.array(hr_preds_all)
    hr_labels_all = np.array(hr_labels_all)
    snr_all       = np.array(snr_all)

    # Aggregate metrics
    n = len(hr_preds_all)
    assert n > 0, "No subjects to evaluate."

    err   = hr_preds_all - hr_labels_all
    abs_e = np.abs(err)
    sq_e  = err ** 2
    rel_e = abs_e / (np.abs(hr_labels_all) + 1e-9)

    mae       = float(np.mean(abs_e))
    mae_se    = float(np.std(abs_e) / np.sqrt(n))
    rmse      = float(np.sqrt(np.mean(sq_e)))
    rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))
    mape      = float(np.mean(rel_e) * 100.0)
    mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

    if n >= 2:
        pearson_r  = float(np.corrcoef(hr_preds_all, hr_labels_all)[0, 1])
        pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
    else:
        pearson_r, pearson_se = float("nan"), float("nan")

    mean_snr    = float(np.mean(snr_all))
    mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

    print(f"\nAggregate Metrics ({model_name}):")
    print(f"  MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
    print(f"  RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
    print(f"  MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
    print(f"  Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
    print(f"  SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

    # Export to per-model subdirectory
    model_output_dir = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(model_output_dir, exist_ok=True)

    metrics_dict = {
        "model":      model_name,
        "n_subjects": n,
        "evaluation_method": "FFT BVP-derived HR",
        "bvp_bandpass_hz":   [0.6, 3.3],
        "aggregate_metrics": {
            "MAE":     {"value": mae,       "se": mae_se,      "unit": "bpm"},
            "RMSE":    {"value": rmse,      "se": rmse_se,     "unit": "bpm"},
            "MAPE":    {"value": mape,      "se": mape_se,     "unit": "%"},
            "Pearson": {"value": pearson_r, "se": pearson_se, "unit": ""},
            "SNR":     {"value": mean_snr,  "se": mean_snr_se, "unit": "dB"},
        },
        "per_subject": [
            {
                "name":                r["name"],
                "predicted_heartrate": r["predicted_heartrate"],
                "label_heartrate":     r["label_heartrate"],
                "heartrate_error":     r["heartrate_error"],
                "snr_db":              r["snr_db"],
            }
            for r in per_subject_results
        ],
    }

    json_path = os.path.join(model_output_dir, "metrics.json")
    with open(json_path, "w") as fh:
        json.dump(metrics_dict, fh, indent=2)
    print(f"\nMetrics saved to: {json_path}")

    # Export ppg_results.csv
    csv_rows = []
    for r in per_subject_results:
        csv_rows.append({
            "name":                r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "label_heartrate":     r["label_heartrate"],
            "heartrate_error":     r["heartrate_error"],
        })

    results_df = pd.DataFrame(csv_rows, columns=[
        "name", "predicted_heartrate", "label_heartrate", "heartrate_error"
    ])

    csv_path = os.path.join(model_output_dir, "ppg_results.csv")
    results_df.to_csv(csv_path, index=False)

    print(f"CSV saved to: {csv_path}")
    print()
    print(results_df.to_string(index=False))

    # Clean up model from GPU
    del model
    torch.cuda.empty_cache()

print(f"\n\nAll models processed. Results in: {OUTPUT_DIR}")

In [ ]:
# convert metric.json to csv

import os
import glob
import json
import pandas as pd

# 1. Khai báo đường dẫn gốc chứa các thư mục model dựa trên ảnh của bạn
ROOT_DIR = OUTPUT_DIR

# 2. Tìm tất cả các file metrics.json nằm trong các thư mục con
json_files = glob.glob(os.path.join(ROOT_DIR, "*", "metrics.json"))

data_rows = []

# 3. Lặp qua từng file JSON để lấy dữ liệu
for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
        model_name = data.get("model", "Unknown")
        n_subjects = data.get("n_subjects", 0)
        metrics = data.get("aggregate_metrics", {})
        
        # Lấy giá trị MAE gốc để làm tiêu chí sắp xếp (Rank)
        mae_raw = metrics.get("MAE", {}).get("value", float('inf'))
        
        # Hàm định dạng chữ theo chuẩn "value +/- se" (làm tròn 2 chữ số)
        def format_metric(m):
            if not m: return ""
            return f"{m.get('value', 0):.2f} +/- {m.get('se', 0):.2f}"

        # Đẩy dữ liệu vào 1 hàng (row)
        row = {
            "model": model_name,
            "# subjects": n_subjects,
            "MAE_raw": mae_raw, # Cột tạm để sort
            "MAE (bpm)": format_metric(metrics.get("MAE")),
            "RMSE (bpm)": format_metric(metrics.get("RMSE")),
            "MAPE (%)": format_metric(metrics.get("MAPE")),
            "Pearson": format_metric(metrics.get("Pearson")),
            "SNR (dB)": format_metric(metrics.get("SNR")),
        }
        data_rows.append(row)

# 4. Chuyển thành DataFrame (bảng)
df = pd.DataFrame(data_rows)

if not df.empty:
    # Sắp xếp bảng theo giá trị MAE thô (từ thấp nhất -> cao nhất)
    df = df.sort_values(by="MAE_raw", ascending=True).reset_index(drop=True)
    
    # Thêm cột 'rank' vào vị trí đầu tiên (bắt đầu từ 1)
    df.insert(0, "rank", df.index + 1)
    
    # Xóa cột 'MAE_raw' vì không cần hiển thị ra CSV
    df = df.drop(columns=["MAE_raw"])
    
    # 5. Xuất ra file CSV
    out_csv_path = os.path.join(ROOT_DIR, "Model_Performance_Metrics.csv")
    df.to_csv(out_csv_path, index=False)
    
    print(f"✅ Đã gom thành công {len(json_files)} file JSON!")
    print(f"✅ File tổng hợp được lưu tại:\n{out_csv_path}\n")
    print("Preview dữ liệu:")
    print(df.head().to_string(index=False))
else:
    print("❌ Không tìm thấy file metrics.json nào trong thư mục!")